# Day 16: Capstone, a Research Assistant

This is the last day, and today we build one complete project from start to finish.

Our research assistant will take a question like "How do I build a reliable AI agent?" and:

1. **plan**: break the question into a few smaller sub-questions
2. **ask you** to approve or edit that plan before spending any money on research
3. **research** every sub-question at the same time, using a small knowledge base
4. **write** a short report from the research notes
5. **review** its own report, and rewrite it if it isn't good enough

```
  START -> plan -> human_review --(one Send per sub-question)--> research -> write -> review --pass--> END
                                                                              ^          |
                                                                              +--revise--+  (at most 2 times)
```

Almost every idea from the course shows up here:

- state, reducers, routing and loops (Days 1 to 3)
- the OpenAI model loaded from `.env` (Day 4)
- searching a knowledge base, RAG style (Day 13)
- structured output for the plan and the review (Day 10)
- pausing for human approval with `interrupt` (Day 8)
- parallel research workers with `Send` (Day 11)
- the write, review, improve loop (Day 10)
- a checkpointer and thread ids (Day 7)
- runtime context for the reader's level (Day 14)
- custom progress messages while streaming (Day 9)
- retry policies (Day 15)

Take your time with this one. Run each cell, read the comments, and when it's working, try changing things.

In [ ]:
%pip install -q -U langgraph langchain-openai langchain-text-splitters python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## Step 1: the knowledge base

The assistant needs something to research from. We'll give it a small library of notes about building AI agents, which is convenient because you know this topic well by now, so you can judge whether the reports are correct.

In a real project, this would be your company documents, or a web search tool.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
library = {
    "agents.md": (
        "An AI agent is an LLM that decides which actions to take in a loop. It calls tools, reads the "
        "results and decides again until the task is done. Agents are flexible but less predictable than "
        "fixed workflows, so use them only when the steps cannot be known in advance."
    ),
    "tools.md": (
        "Tools let an LLM use calculators, APIs and databases. Each tool needs a clear name, typed "
        "arguments and a good description. Tool errors should be returned to the model as messages so it "
        "can recover instead of crashing the app."
    ),
    "memory.md": (
        "Short-term memory keeps the current conversation using a checkpointer and a thread id. "
        "Long-term memory stores facts about a user across conversations in a store. Long histories "
        "should be trimmed or summarized to control cost."
    ),
    "reliability.md": (
        "Reliable agents use step limits to stop endless loops, retry policies for flaky APIs, "
        "structured output for decisions, and human approval before risky actions such as payments "
        "or sending emails. Tracing tools like LangSmith help debug each step."
    ),
    "evaluation.md": (
        "Evaluate agents with test questions and expected behaviours. An evaluator model can grade "
        "answers for correctness and tone. Track cost, latency and success rate over time."
    ),
    "rag.md": (
        "Retrieval-augmented generation finds relevant document chunks with embeddings and gives them "
        "to the model as context. Good RAG systems cite sources, say 'I don't know' when context is "
        "missing, and rewrite unclear queries."
    ),
}

In [ ]:
documents = [Document(page_content=text, metadata={"source": name}) for name, text in library.items()]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
chunks = splitter.split_documents(documents)

retriever = InMemoryVectorStore.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": 2})
print(len(chunks), "chunks ready to search")

## Step 2: the state, the context and the "forms"

First, the state. Notice `notes` has a reducer: several research workers will write to it at the same time (that's the lesson from Days 2 and 11).

In [ ]:
import operator
from typing import TypedDict, Annotated

In [ ]:
class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    notes: Annotated[list[dict], operator.add]    # filled in by the parallel workers
    report: str
    feedback: str
    verdict: str
    revisions: int

Next, the runtime context. Who is going to read this report? A beginner needs a different report from a busy manager. That's a setting for this run, not part of the conversation, so it goes in the context (Day 14).

In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class Context:
    reader_level: str = "beginner"

And two structured-output "forms" (Day 10): one for the plan, one for the review.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
class Plan(BaseModel):
    sub_questions: list[str] = Field(description="2 to 4 focused sub-questions that together answer the main question")

In [ ]:
class Review(BaseModel):
    verdict: Literal["pass", "revise"] = Field(description="pass if the report is clear, well structured and based on the notes")
    feedback: str = Field(description="specific things to fix if revise, otherwise empty")

In [ ]:
planner = llm.with_structured_output(Plan)
reviewer = llm.with_structured_output(Review)

MAX_REVISIONS = 2

## Step 3: the nodes

Each node does one small job. Read them one at a time.

**Plan.** Break the big question into a few smaller ones. `get_stream_writer()` lets us send a progress message (Day 9).

In [ ]:
from langgraph.config import get_stream_writer

In [ ]:
def plan(state: ResearchState) -> dict:
    get_stream_writer()({"status": "Planning the research..."})

    result = planner.invoke(f"Break this research question into sub-questions: {state['question']}")
    return {"sub_questions": result.sub_questions[:4]}       # never more than 4

**Human review.** Pause and show the plan to a person (Day 8). They can reply "approve", or send back their own list of sub-questions. Then we hand out one research job per sub-question, using `Send` inside a `Command` (Days 3 and 11).

In [ ]:
from langgraph.types import interrupt, Send, Command

In [ ]:
def human_review(state: ResearchState) -> Command[Literal["research"]]:
    answer = interrupt({
        "message": "Approve this plan? Reply 'approve', or send your own list of sub-questions.",
        "sub_questions": state["sub_questions"],
    })

    # If the person sent a list, use theirs. Otherwise keep the original plan.
    if isinstance(answer, list):
        sub_questions = answer
    else:
        sub_questions = state["sub_questions"]

    research_jobs = [Send("research", {"sub_question": q}) for q in sub_questions]
    return Command(update={"sub_questions": sub_questions}, goto=research_jobs)

**Research.** Each worker gets one sub-question, searches the knowledge base, and writes a short note with sources (Day 13). Many of these run at the same time.

In [ ]:
def research(job: dict) -> dict:
    sub_question = job["sub_question"]
    get_stream_writer()({"status": f"Researching: {sub_question}"})

    found = retriever.invoke(sub_question)
    context = "\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in found)

    prompt = (
        "Using ONLY this context, answer in 2 sentences and keep the [source] tags.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {sub_question}"
    )
    note = llm.invoke(prompt).content
    return {"notes": [{"sub_question": sub_question, "note": note}]}

**Write.** Turn all the notes into a short report for the right reader. If the reviewer sent feedback, include the previous draft and the feedback, so the writer improves it rather than starting from scratch.

In [ ]:
from langgraph.runtime import Runtime

In [ ]:
def write(state: ResearchState, runtime: Runtime[Context]) -> dict:
    if state.get("feedback"):
        get_stream_writer()({"status": "Revising the report..."})
    else:
        get_stream_writer()({"status": "Writing the report..."})

    notes = "\n".join(f"- Q: {n['sub_question']}\n  A: {n['note']}" for n in state["notes"])

    prompt = (
        f"Write a short research report (at most 180 words) answering: {state['question']}\n"
        f"The reader is: {runtime.context.reader_level}. Use markdown headings and bullet points.\n"
        f"Use only these notes, and keep their [source] tags:\n{notes}"
    )
    if state.get("feedback"):
        prompt += f"\n\nYour previous draft:\n{state['report']}\n\nFix this feedback: {state['feedback']}"

    return {"report": llm.invoke(prompt).content}

**Review.** A second LLM call grades the report, like an editor (Day 10).

In [ ]:
def review(state: ResearchState) -> dict:
    get_stream_writer()({"status": "Reviewing the report..."})

    result = reviewer.invoke(
        f"Question: {state['question']}\n\nReport:\n{state['report']}\n\n"
        "Is the report clear, well structured, and does it cite sources?"
    )
    return {
        "verdict": result.verdict,
        "feedback": result.feedback,
        "revisions": state.get("revisions", 0) + 1,
    }

**After review.** The router for our loop. Stop if it passed, or if we've already revised enough times (always cap your loops, Day 3).

In [ ]:
def after_review(state: ResearchState) -> Literal["write", "__end__"]:
    if state["verdict"] == "pass":
        return END
    if state["revisions"] >= MAX_REVISIONS:
        return END
    return "write"

## Step 4: putting the graph together

Notice how few edges there are. `human_review` decides its next step with `Command`, so it needs no outgoing edge. And `research` to `write` is a normal edge, but because all the research jobs run in the same step, `write` waits until **every** one of them is finished.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy

In [ ]:
retry = RetryPolicy(max_attempts=3)

In [ ]:
builder = StateGraph(ResearchState, context_schema=Context)

In [ ]:
builder.add_node("plan", plan, retry_policy=retry)
builder.add_node("human_review", human_review)
builder.add_node("research", research, retry_policy=retry)
builder.add_node("write", write, retry_policy=retry)
builder.add_node("review", review)

In [ ]:
builder.add_edge(START, "plan")
builder.add_edge("plan", "human_review")
builder.add_edge("research", "write")
builder.add_edge("write", "review")
builder.add_conditional_edges("review", after_review)

In [ ]:
assistant = builder.compile(checkpointer=InMemorySaver())
show_graph(assistant)

## Step 5: run it, and stop at the approval

We'll stream two modes at once (Day 9): our custom progress messages, and node updates, so we can catch the moment the graph pauses for us.

This little helper function does the printing, so we can reuse it.

In [ ]:
def run(graph_input, config, context):
    for mode, data in assistant.stream(graph_input, config, context=context, stream_mode=["custom", "updates"]):
        if mode == "custom":
            print(data["status"])
        elif "__interrupt__" in data:
            paused = data["__interrupt__"][0].value
            print()
            print(paused["message"])
            for number, sub_question in enumerate(paused["sub_questions"], start=1):
                print(f"   {number}. {sub_question}")

In [ ]:
config = {"configurable": {"thread_id": "research-1"}}
reader = Context(reader_level="beginner")

run({"question": "How do I build a reliable AI agent?", "revisions": 0}, config, reader)

### Predict before you run

The graph is now paused. When we approve, which progress messages do you expect to see, and in what order?

<details>
<summary>Click to see the answer</summary>

One "Researching: ..." line per sub-question (their order can vary, because they run in parallel), then "Writing the report...", then "Reviewing the report...".

If the reviewer asks for changes, you'll also see "Revising the report..." and another "Reviewing the report...", up to the revision limit.
</details>

## Step 6: approve the plan and let it finish

You have two choices here:

- approve as it is: `Command(resume="approve")`
- send your own plan: `Command(resume=["What makes agents fail?", "How do memory and tools help?"])`

In [ ]:
run(Command(resume="approve"), config, reader)

In [ ]:
final = assistant.get_state(config).values

print(f"Revisions: {final['revisions']}, final verdict: {final['verdict']}")
print()
print(final["report"])

## Step 7: look at what happened

It's always worth looking at the intermediate results, not just the final report. This is how you find out *why* a report is good or bad.

In [ ]:
print("What each research worker found:")
for note in final["notes"]:
    print()
    print("Q:", note["sub_question"])
    print("A:", note["note"])

In [ ]:
print("Checkpoints saved for this thread:", len(list(assistant.get_state_history(config))))

## Step 8: same graph, different reader

Same question style, a new thread, and a different reader in the runtime context. No code changes.

In [ ]:
config = {"configurable": {"thread_id": "research-2"}}
manager = Context(reader_level="a busy manager who cares about risks and costs")

run({"question": "What should we check before putting an AI agent into production?", "revisions": 0}, config, manager)

In [ ]:
run(Command(resume="approve"), config, manager)

print()
print(assistant.get_state(config).values["report"])

### Your turn

Run the assistant once more, but this time **edit the plan** instead of approving it. Replace the list below with your own sub-questions, and see how the report changes.

In [ ]:
config = {"configurable": {"thread_id": "research-3"}}

run({"question": "How should a beginner start learning about AI agents?", "revisions": 0}, config, reader)

In [ ]:
my_plan = [
    "What is an AI agent?",
    "Why do agents need tools and memory?",
]      # write your own sub-questions here

run(Command(resume=my_plan), config, reader)

print()
print(assistant.get_state(config).values["report"])

## Common mistakes in a project like this

**Forgetting the reducer on `notes`.** With parallel research workers, you'd get an `InvalidUpdateError` the moment two workers finish. (Day 2, Day 11.)

**Side effects before `interrupt()`.** If `human_review` sent an email before pausing, the email would go out twice: once on the first run and once on resume. (Day 8.)

**No revision limit.** A strict reviewer and a weak writer can loop forever. `MAX_REVISIONS` keeps the cost under control. (Days 3 and 10.)

**Resuming on a different thread.** The approval must use the same `config` as the run that paused. (Day 8.)

**Judging the system only by the final report.** When the report is weak, look at the notes in Step 7. Usually the problem is upstream: the plan was poor, or the search returned the wrong chunks.

## Ideas to make it your own

1. **Real web search**: replace the retriever in `research` with a search tool, for example Tavily.
2. **Remember the reader**: save each user's preferred reader level in a Store (Day 14), so they don't have to set it every time.
3. **A chat interface**: wrap `assistant` in FastAPI or Streamlit, and stream the `"messages"` mode so the report types itself out (Days 9 and 15).
4. **Your own documents**: load your PDFs with `PyPDFLoader` and build the library from them.
5. **Deploy it**: move the code into `research_assistant.py` and run it with `langgraph dev` (Day 15).

## You've finished the course

Let's look back at how far you've come in 16 days:

- **Days 1 to 3**: state, nodes, edges, reducers, branches, loops and `Command`, the whole core of LangGraph, with plain Python.
- **Days 4 to 6**: real LLMs, chat messages, chatbots, tools and agents.
- **Days 7 to 9**: memory, human approval, time travel, streaming and async.
- **Days 10 to 12**: structured output, workflow patterns, `Send`, subgraphs and teams of agents.
- **Days 13 to 15**: RAG over your own documents, long-term memory, and getting ready for production.
- **Day 16**: one complete project that uses almost all of it.

The best next step is to build something small for a real problem you actually have. An assistant for your own notes, a helper for a repetitive task at work, a study bot for your exams. Keep it small, add tracing, and improve it one piece at a time.

Well done for getting to the end.